# Results and diagnostics

Report JECAM first, then California carrot, Denmark cabbage, non-JECAM potato, and the combined pool. JECAM is the primary benchmark. Its Madagascar site provides the closest documented comparison to Paoay’s highland vegetable farming.

The reporting scope was revised after score review. It retains potato from the non-JECAM carrot/potato campaign and excludes non-JECAM cabbage and carrot.

Batch 1 uses saved parcel predictions, training-majority baselines and usable coverage from the frozen run. The report shows parcel and pixel counts, F1 with 95% intervals, and the paired comparison with majority. Combined intervals condition on observed campaign and reference-crop counts.

The settings cell selects the frozen Batch 1 run when available. To review a different saved Testing 01 handoff, set `scores_handoff`. Standard diagnostics are available for other saved runs.

[Run order and reruns](../../docs/workflow.md#training-and-external-testing)


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from crop_pipeline.training import notebook_workflow as workflow


In [ ]:
from tempfile import mkdtemp
from IPython.display import Markdown, display

from crop_pipeline.training import external_report

scores_handoff = ""  # Optional saved Testing 01 scores.json for standard diagnostics
batch1_run = root / "pipeline_outputs/testing/run_81a0d0a2812e"
batch1_available = not scores_handoff and (batch1_run / "scores.json").is_file()

if batch1_available:
    report_folder = Path(mkdtemp(prefix=f"{batch1_run.name}_batch1_revised_", dir=batch1_run.parent))
    batch1_output = report_folder / "report"
    external_report.report(batch1_run, batch1_output, revised=True)
    display(Markdown((batch1_output / "report.md").read_text()))
    print("Batch 1 tables:", batch1_output)
else:
    report_handoff = workflow.report(root, scores_handoff or None)
    display(Markdown((report_handoff.parent / "results.md").read_text()))
    print("Tables:", report_handoff.parent)


## Evidence and interpretation

Report parcel support and coverage alongside scores. External results describe the sampled test population. See [research basis and limits](../../docs/research_basis.md) for sources and project-specific choices.


## Read the parcel errors

Rows are reference crops; columns are predictions. Counts on the diagonal are correct. Inspect missing reference support before interpreting zeros. Change `population` to a saved supplement or combined population for a separate view.


In [ ]:
from crop_pipeline.windows import seasons
from crop_pipeline.notebook_display import show_table
from crop_pipeline.visualization.demo import plot_confusion
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
population = "jecam"
if batch1_available:
    metrics = seasons.read_table(batch1_output / "parcel_metrics.csv")
    show_table([row for row in metrics if row["population"] == population
                and row["predictor"] == "model" and row["crop"] and int(row["support"])],
               title="Crop scores, coverage and 95% intervals")
    cells = seasons.read_table(batch1_output / "confusion_matrices.csv")
    confusion_rows = [{"test_population": row["population"], "cohort": row["model"],
                       "actual": row["actual"], "predicted": row["predicted"], "parcels": row["count"]}
                      for row in cells if row["predictor"] == "model"]
else:
    show_table(seasons.read_table(report_handoff.parent / "per_crop.csv"), title="Crop scores and coverage")
    confusion_rows = seasons.read_table(report_handoff.parent / "confusion_matrices.csv")
figure = plot_confusion(confusion_rows, population)
display(figure)
plt.close(figure)


## Figures

External-test macro-F1 beside validation, the training-majority baseline, per-crop F1 and confusion matrices. The 0.50 line is the agreed target. Test scores are never used to choose a model.

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from crop_pipeline.visualization import workflow_figures as figures


def show_and_save(figure, folder, name):
    """Display inline and keep a PNG beside the run outputs for reports and slides."""
    print("Saved figure:", figures.save_figure(figure, folder, name))
    display(figure)
    plt.close(figure)

import json
from pathlib import Path

use_batch1_figures = globals().get("batch1_available", False)
reports = batch1_run / "reports" if use_batch1_figures else report_handoff.parent.parent
population = globals().get("population", "jecam")
scope_reports = reports if population == "jecam" else reports / population
if use_batch1_figures:
    test_scores = json.loads((batch1_output / "figure_scores.json").read_text())[population]
    paired = seasons.read_table(batch1_output / "paired_macro_f1.csv")
    baseline = {row["model"]: float(row["majority_macro_f1"]) if row["majority_macro_f1"] else None
                for row in paired if row["population"] == population}
else:
    test_scores = json.loads((scope_reports / "test_scores.json").read_text())
    baseline = figures.baseline_macro_f1(json.loads((scope_reports / "majority_baseline.json").read_text()))
models_handoff = Path(json.loads((reports.parent / "scores.json").read_text())["models_handoff"])
validation_scores = figures.load_validation_scores(models_handoff.parent / "models.json")
figure_folder = (batch1_output if use_batch1_figures else scope_reports) / "figures"
show_and_save(figures.draw_macro_f1(test_scores, f"Validation vs {population} external-test macro-F1", baseline=baseline,
                                    comparison=validation_scores), figure_folder, "test_macro_f1")
show_and_save(figures.draw_per_crop_f1(test_scores, "Per-crop external-test F1 (target ≥ 0.50)"),
              figure_folder, "test_per_crop_f1")
show_and_save(figures.draw_confusion_matrices(test_scores, "External-test confusion matrices (parcels; shade = row share)"),
              figure_folder, "test_confusion_matrices")